# Weekend vs. Weekday Shopping: Sample to Population Inference

**The Question:** Is there a statistically significant difference between average
weekend and weekday shopper spending (testing $H_0: \mu_{\text{weekend}} = \mu_{\text{weekday}}$)?

**The Data:** UCI *Online Retail* dataset (Chen, D., 2015, CC BY 4.0),
real, timestamped transaction logs from a UK online retailer (Dec 2010 to Dec 2011).

**Core Concept:** Real-world studies analyze samples, not full populations.
Using this dataset, we see how reliably sample conclusions match the actual
population results when testing our null hypothesis.

**Analysis Plan:**
1. **Initial Sample:** Draw a sample of weekend and weekday transactions,
   assume equal variance, and run a standard pooled $t$-test to test $H_0$.
2. **Outlier Filtering:** Restrict data to typical buyers using the IQR rule
   ($Q_3 + 1.5 \times \text{IQR}$) to remove untypical buyers, then re-test $H_0$ on the trimmed sample.
3. **Population Check:** Compute true population means across the entire
   dataset to evaluate whether $H_0$ holds in the population.
4. **Model Comparison:** Evaluate $H_0$ using **Welch's $t$-test** alongside
   the pooled test (relaxing the equal variance assumption) to see if variance
   differences alter our conclusions about the means.

In [3]:
import numpy as np
import pandas as pd
from scipy import stats

## 0. Understand the data


In [4]:
url = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
raw = pd.read_csv(url, encoding="latin1", dtype={"CustomerID": str})
raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


**How big is the data?** 

In [5]:
raw.shape

(541909, 8)

**What is the data's type?**

In [6]:
raw.dtypes

InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID      object
Country         object
dtype: object

### Label each row Weekend or Weekday

`InvoiceDate` is currently `object` (plain text), not a real date. It needs to be converted to a datetime first.
Once converted, `.dt.day_name()` pulls out each row's actual day (e.g. "Monday", "Sunday"), which then gets
collapsed into the two categories: Weekend and Weekday.

In [7]:
# Convert InvoiceDate string representations into datetime objects using the explicit timestamp format
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"], format="%m/%d/%Y %H:%M")

# Extract the corresponding day of the week (e.g., 'Monday', 'Sunday') as a string series
raw["DayOfWeek"] = raw["InvoiceDate"].dt.day_name()

# Categorize days into binary classifications ('Weekend' vs. 'Weekday') based on day name
day_types = []
for day in raw["DayOfWeek"]:
    if day == "Saturday" or day == "Sunday":
        day_types.append("Weekend")
    else:
        day_types.append("Weekday")
raw["DayType"] = day_types

# Display summary frequencies across both 'Weekend' and 'Weekday' categories
raw["DayType"].value_counts()

DayType
Weekday    477534
Weekend     64375
Name: count, dtype: int64

#### Check the type of invoices

Check what prefixes appear in the data.

In [9]:
invoice = raw["InvoiceNo"].astype(str)
first_char = invoice.str[0]
first_char.value_counts()

InvoiceNo
5    532618
C      9288
A         3
Name: count, dtype: int64

Three types of invoice numbers exist: ordinary
numeric invoices (532,618 of them), invoices starting with **C** (9,288), and a small number
starting with **A** (just 3). 

In [8]:
raw[raw["InvoiceNo"].astype(str).str.startswith("A")]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,DayOfWeek,DayType
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,Friday,Weekday
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,Friday,Weekday
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,Friday,Weekday


These are internal **accounting adjustments** ("Adjust bad debt"), not real purchases. Nothing about these rows reflects shopping behavior. Therefore, these will not be counted as orders.

In [10]:
raw[raw["InvoiceNo"].astype(str).str.startswith("C")].head(5)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,DayOfWeek,DayType
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527,United Kingdom,Wednesday,Weekday
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311,United Kingdom,Wednesday,Weekday
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548,United Kingdom,Wednesday,Weekday
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom,Wednesday,Weekday
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548,United Kingdom,Wednesday,Weekday


These are **cancellations**: each of them has a negative `Quantity`. Again, these do not reflect purchases.

In [11]:
ordinary_purchase = ~invoice.str.startswith(("C", "A"))
clean = raw[ordinary_purchase]
clean = clean[(clean["Quantity"] > 0) & (clean["UnitPrice"] > 0)]
clean.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,DayOfWeek,DayType
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Wednesday,Weekday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Wednesday,Weekday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Wednesday,Weekday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Wednesday,Weekday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Wednesday,Weekday
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,Wednesday,Weekday
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,Wednesday,Weekday
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,Wednesday,Weekday
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,Wednesday,Weekday
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom,Wednesday,Weekday


#### Collapse line items into one row per order

We note that the `clean` data set has one row per **product**, not one row per **order**: a single order can span
several rows, all sharing the same `InvoiceNo` (see the first 7 rows above, all `InvoiceNo=536365`).
We only care about **orders**, not individual products, so these product-level rows
need to be grouped and summed by `InvoiceNo` into one row per order, each with a single total dollar value.

In [13]:
# Calculate total spend for each product line item
clean["LineTotal"] = clean["Quantity"] * clean["UnitPrice"]

# Group by InvoiceNo and aggregate line items into one row per order
orders = (
    clean.groupby("InvoiceNo")
    .agg(
        OrderValue=("LineTotal", "sum"),  # Sum item totals to get overall order value
        InvoiceDate=("InvoiceDate", "first"),  # Keep the order timestamp
        DayType=("DayType", "first"),  # Keep the day classification ('Weekday'/'Weekend')
    )
    .reset_index()  # Return InvoiceNo to a standard column
)

# View the first 10 orders
orders.head(10)

,InvoiceNo,OrderValue,InvoiceDate,DayType
0,536365,139.12,2010-12-01 08:26:00,Weekday
1,536366,22.20,2010-12-01 08:28:00,Weekday
2,536367,278.73,2010-12-01 08:34:00,Weekday
3,536368,70.05,2010-12-01 08:34:00,Weekday
4,536369,17.85,2010-12-01 08:35:00,Weekday
5,536370,855.86,2010-12-01 08:45:00,Weekday
6,536371,204.00,2010-12-01 09:00:00,Weekday
7,536372,22.20,2010-12-01 09:01:00,Weekday
8,536373,259.86,2010-12-01 09:02:00,Weekday
9,536374,350.40,2010-12-01 09:09:00,Weekday


## Step 1 — Small Sample Analysis

In real-world scenarios, full-dataset access is not always available at the outset—we often have to make decisions using only a smaller sample. To simulate these practical constraints, we draw an initial sample of $n_1 = n_2 = 500$ orders per group to conduct our initial analysis and check sample distributions.

In [17]:
# Extract OrderValue arrays for Weekend and Weekday orders separately
# .loc filters rows by condition (DayType) and selects the specified column ('OrderValue')
weekend_orders = orders.loc[orders["DayType"] == "Weekend", "OrderValue"]
weekday_orders = orders.loc[orders["DayType"] == "Weekday", "OrderValue"]

# Set up a reproducible random number generator
rng = np.random.default_rng(2)

# Randomly draw 500 samples without replacement from each group
weekend = rng.choice(weekend_orders.values, size=500, replace=False)
weekday = rng.choice(weekday_orders.values, size=500, replace=False)

# ==========================================
# 1. NumPy Built-in Calculations
# ==========================================
n1_np, n2_np = len(weekend), len(weekday)
x1_np, x2_np = weekend.mean(), weekday.mean()
s1_np, s2_np = weekend.std(ddof=1), weekday.std(ddof=1)

# ==========================================
# 2. Manual Mathematical Calculations
# ==========================================
n1_man = len(weekend)
n2_man = len(weekday)

# Sample Means: sum / count
x1_man = sum(weekend) / n1_man
x2_man = sum(weekday) / n2_man

# Sample Standard Deviations: sqrt( sum((x - mean)^2) / (n - 1) )
s1_man = (sum((x - x1_man) ** 2 for x in weekend) / (n1_man - 1)) ** 0.5
s2_man = (sum((x - x2_man) ** 2 for x in weekday) / (n2_man - 1)) ** 0.5

# ==========================================
# 3. Print Both Results Side-by-Side
# ==========================================
print("--- NUMPY METHOD ---")
print(f"Weekend: n={n1_np}, mean=${x1_np:.2f}, sd=${s1_np:.2f}")
print(f"Weekday: n={n2_np}, mean=${x2_np:.2f}, sd=${s2_np:.2f}\n")

print("--- MANUAL METHOD ---")
print(f"Weekend: n={n1_man}, mean=${x1_man:.2f}, sd=${s1_man:.2f}")
print(f"Weekday: n={n2_man}, mean=${x2_man:.2f}, sd=${s2_man:.2f}")

--- NUMPY METHOD ---
Weekend: n=500, mean=$368.86, sd=$382.24
Weekday: n=500, mean=$458.64, sd=$686.09

--- MANUAL METHOD ---
Weekend: n=500, mean=$368.86, sd=$382.24
Weekday: n=500, mean=$458.64, sd=$686.09


#### What `s1`/`s2` are computing?

$$S = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar x)^2}$$

For each order value $x_i$ in a group, $(x_i-\bar x)$ is how far that order sits from its group's
own mean. Summing across all $n$ orders and dividing by $n-1$ gives the
average squared deviation, an unbiased estimate of the population variance; the $n-1$ correction
(rather than $n$) accounts for the fact that one degree of freedom is already used up computing
$\bar x$ from the same data. 

This is exactly what `.std(ddof=1)` computes: `ddof=1` is what makes us divide by
$n-1$ instead of the default $n$.

#### $H_0: \mu_{\text{weekend}} = \mu_{\text{weekday}} \quad H_a: \mu_{\text{weekend}} \neq \mu_{\text{weekday}}$

We perform a two-sample $t$-test to determine if there is a statistically significant difference in average order values between weekend and weekday purchases. Under the null hypothesis ($H_0$), we assume that both population means are equal ($\mu_{\text{weekend}} = \mu_{\text{weekday}}$), meaning whether an order is placed on a weekend or a weekday has no effect on average spending.

To evaluate this hypothesis, we compute two variants of the test based on different assumptions about variance:
* **Pooled $t$-test:** Assumes the two population groups share an equal variance ($\sigma_1^2 = \sigma_2^2$).
* **Welch's $t$-test:** Does not assume equal variances ($\sigma_1^2 \neq \sigma_2^2$) and uses unequal variances along with adjusted degrees of freedom to account for sample variance disparities.

In [21]:
# ==========================================
# 1. Pooled Two-Sample t-Test (Assumes Equal Variances)
# ==========================================
# Calculate pooled variance, standard error, t-statistic, and p-value
sp2 = ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)  # Pooled variance
sp = np.sqrt(sp2)
se = sp * np.sqrt(1 / n1 + 1 / n2)
t_stat = (x1 - x2) / se
dof = n1 + n2 - 2
p_value = 2 * stats.t.sf(abs(t_stat), dof)

print(f"Pooled variance (Sp^2): {sp2:.4f}")
print(f"t-statistic:            {t_stat:.4f}")
print(f"Degrees of freedom:     {dof}")
print(f"Attained p-value:       {p_value:.4g}")

# Verification: Checking manual Pooled t-test calculation against SciPy's built-in function
t_check, p_check = stats.ttest_ind(weekend, weekday, equal_var=True)
print(
    f"scipy check: t={t_check:.4f}, p={p_check:.4f}  (matches manual calc:"
    f" {np.isclose(t_stat, t_check)})"
)


# ==========================================
# 2. Welch's t-Test (Does NOT Assume Equal Variances)
# ==========================================
# Calculate unpooled standard error, t-statistic, Welch-Satterthwaite degrees of freedom, and p-value
s1, s2 = weekend.std(ddof=1), weekday.std(ddof=1)
se_welch = np.sqrt(s1**2 / n1 + s2**2 / n2)
t_welch = (x1 - x2) / se_welch
v = ((s1**2 / n1 + s2**2 / n2) ** 2) / (
    ((s1**2 / n1) ** 2 / (n1 - 1)) + ((s2**2 / n2) ** 2 / (n2 - 1))
)
p_welch = 2 * stats.t.sf(abs(t_welch), v)

print(f"\nWelch's t-test:")
print(f"t-statistic:            {t_welch:.4f}")
print(f"Degrees of freedom:     {v:.2f}")
print(f"Attained p-value:       {p_welch:.4g}")

# Verification: Checking manual Welch's t-test calculation against SciPy's built-in function
t_welch_check, p_welch_check = stats.ttest_ind(weekend, weekday, equal_var=False)
print(
    f"scipy check: t={t_welch_check:.4f}, p={p_welch_check:.4f}  (matches"
    f" manual calc: {np.isclose(t_welch, t_welch_check)})"
)

Pooled variance (Sp^2): 308410.6311
t-statistic:            -2.5561
Degrees of freedom:     998
Attained p-value:       0.01073
scipy check: t=-2.5561, p=0.0107  (matches manual calc: True)

Welch's t-test:
t-statistic:            -2.5561
Degrees of freedom:     781.55
Attained p-value:       0.01077
scipy check: t=-2.5561, p=0.0108  (matches manual calc: True)


### Variance problem

Weekday $sd=\$686.09$ is close to the weekday mean ($\$458.64$). For a variable bounded below by zero, such as spending, a standard deviation approaching or exceeding the mean is inconsistent with a roughly normal, well-behaved distribution.

The cause is that this dataset combines ordinary shoppers with occasional large orders. A small number of such orders inflate both the mean and the variance substantially, and the $p$-value obtained in our previous calculation (0.01073) is directly affected by them. Because the variance is this large, we would like to restrict the distribution to typical buyers.

## Step 2 — Filter to Typical Shoppers

To focus our analysis on typical shopping behavior, high-value purchases are filtered out using the standard IQR threshold ($Q_3 + 1.5 \times \text{IQR}$). To ensure a consistent baseline for comparison, we calculate one common threshold from the full dataset and apply it equally to both weekday and weekend orders.

In [26]:
sample_values = np.concatenate([weekend, weekday])   # the n1+n2 = 1000 orders from the sample
q1, q3 = np.percentile(sample_values, [25, 75])
cutoff = q3 + 1.5 * (q3 - q1)

# apply the SAME shared cutoff to each group separately, so groups stay identifiable
weekend_trimmed = weekend[weekend <= cutoff]
weekday_trimmed = weekday[weekday <= cutoff]
sample_trimmed = np.concatenate([weekend_trimmed, weekday_trimmed])

q1_trimmed, q3_trimmed = np.percentile(sample_trimmed, [25, 75])

print(f"Mean before trimming: ${sample_values.mean():.2f}")
print(f"Mean after trimming:  ${sample_trimmed.mean():.2f}")
print(f"Q1 (trimmed):   ${q1_trimmed:.2f}")
print(f"Q3 (trimmed):   ${q3_trimmed:.2f}")
print(f"Cutoff: {cutoff:.2f}")
print(f"\nRemoved from weekend: {(weekend>cutoff).sum()}/{len(weekend)}")
print(f"Removed from weekday: {(weekday>cutoff).sum()}/{len(weekday)}")


Mean before trimming: $413.75
Mean after trimming:  $298.04
Q1 (trimmed):   $147.50
Q3 (trimmed):   $396.80
Cutoff: 921.86

Removed from weekend: 26/500
Removed from weekday: 56/500


#### Apply the cutoff and rerun the test


In [27]:
n1, n2 = len(weekend_trimmed), len(weekday_trimmed)
x1, x2 = weekend_trimmed.mean(), weekday_trimmed.mean()
s1, s2 = weekend_trimmed.std(ddof=1), weekday_trimmed.std(ddof=1)

print(f"Weekend (trimmed): n={n1}, mean=${x1:.2f}, sd=${s1:.2f}")
print(f"Weekday (trimmed): n={n2}, mean=${x2:.2f}, sd=${s2:.2f}")

# H0: mu_weekend = mu_weekday   Ha: mu_weekend != mu_weekday

# Pooled test (assumes equal variances)
sp2 = ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)
sp = np.sqrt(sp2)
se = sp * np.sqrt(1/n1 + 1/n2)
t_stat = (x1 - x2) / se
dof = n1 + n2 - 2
p_value = 2 * stats.t.sf(abs(t_stat), dof)

print(f"\nPooled variance (Sp^2): {sp2:.4f}")
print(f"t-statistic:            {t_stat:.4f}")
print(f"Degrees of freedom:     {dof}")
print(f"Attained p-value:       {p_value:.4g}")

t_check, p_check = stats.ttest_ind(weekend_trimmed, weekday_trimmed, equal_var=True)
print(f"scipy check: t={t_check:.4f}, p={p_check:.4f}  (matches manual calc: {np.isclose(t_stat, t_check)})")

# Welch's test (does not assume equal variances)
se_welch = np.sqrt(s1**2/n1 + s2**2/n2)
t_welch = (x1 - x2) / se_welch
v = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))
p_welch = 2 * stats.t.sf(abs(t_welch), v)

print(f"\nWelch's t-test:")
print(f"t-statistic:            {t_welch:.4f}")
print(f"Degrees of freedom:     {v:.2f}")
print(f"Attained p-value:       {p_welch:.4g}")

t_welch_check, p_welch_check = stats.ttest_ind(weekend_trimmed, weekday_trimmed, equal_var=False)
print(f"scipy check: t={t_welch_check:.4f}, p={p_welch_check:.4f}  (matches manual calc: {np.isclose(t_welch, t_welch_check)})")

Weekend (trimmed): n=474, mean=$304.61, sd=$198.28
Weekday (trimmed): n=444, mean=$291.03, sd=$203.79

Pooled variance (Sp^2): 40386.5943
t-statistic:            1.0227
Degrees of freedom:     916
Attained p-value:       0.3067
scipy check: t=1.0227, p=0.3067  (matches manual calc: True)

Welch's t-test:
t-statistic:            1.0218
Degrees of freedom:     908.17
Attained p-value:       0.3072
scipy check: t=1.0218, p=0.3072  (matches manual calc: True)


Much better-behaved: means (\$304.61  vs  \$291.03) and SDs (\$198.28, \$203.79) are now on comparable
scales, and $(mean-1\,sd)$ is safely positive for both groups. The $t$-test's assumptions are far more
defensible here. On this sample, the difference is **not statistically significant** (pooled
$p\approx0.3067$, Welch $p\approx0.3072$ — the two nearly agree, since $n_1\approx n_2$ here).

## Step 3 — Full Population Analysis

In practice, researchers must rely on collected samples, such as the $n = 500$ per group subset used previously. However, because our dataset contains the complete transactional history, we can analyze the full population directly. Rather than estimating parameters, we can calculate the exact population parameters and run our hypothesis test across all available orders.

In [33]:
# --- Untrimmed (raw) full population, for comparison ---
weekend_pop_raw = orders.loc[orders.DayType == "Weekend", "OrderValue"].values
weekday_pop_raw = orders.loc[orders.DayType == "Weekday", "OrderValue"].values

n1_raw, n2_raw = len(weekend_pop_raw), len(weekday_pop_raw)
x1_raw, x2_raw = weekend_pop_raw.mean(), weekday_pop_raw.mean()
s1_raw, s2_raw = weekend_pop_raw.std(ddof=1), weekday_pop_raw.std(ddof=1)

print(f"Weekend population (raw): n={n1_raw:,}, mean=${x1_raw:.2f}, sd=${s1_raw:.2f}")
print(f"Weekday population (raw): n={n2_raw:,}, mean=${x2_raw:.2f}, sd=${s2_raw:.2f}")

# Pooled test (assumes equal variances)
sp2_raw = ((n1_raw - 1) * s1_raw**2 + (n2_raw - 1) * s2_raw**2) / (n1_raw + n2_raw - 2)
sp_raw = np.sqrt(sp2_raw)
se_raw = sp_raw * np.sqrt(1/n1_raw + 1/n2_raw)
t_stat_raw = (x1_raw - x2_raw) / se_raw
dof_raw = n1_raw + n2_raw - 2
p_value_raw = 2 * stats.t.sf(abs(t_stat_raw), dof_raw)

print(f"\nPooled variance (Sp^2): {sp2_raw:.4f}")
print(f"t-statistic:            {t_stat_raw:.4f}")
print(f"Degrees of freedom:     {dof_raw}")
print(f"Attained p-value:       {p_value_raw:.6f}  ({p_value_raw:.3g} to 3 s.f.)")

t_check_raw, p_check_raw = stats.ttest_ind(weekend_pop_raw, weekday_pop_raw, equal_var=True)
print(f"scipy check: t={t_check_raw:.4f}, p={p_check_raw:.7f}  (matches manual calc: {np.isclose(t_stat_raw, t_check_raw)})")

# Welch's test (does not assume equal variances)
se_welch_raw = np.sqrt(s1_raw**2/n1_raw + s2_raw**2/n2_raw)
t_welch_raw = (x1_raw - x2_raw) / se_welch_raw
v_raw = (s1_raw**2/n1_raw + s2_raw**2/n2_raw)**2 / ((s1_raw**2/n1_raw)**2/(n1_raw-1) + (s2_raw**2/n2_raw)**2/(n2_raw-1))
p_welch_raw = 2 * stats.t.sf(abs(t_welch_raw), v_raw)

print(f"\nWelch's t-test:")
print(f"t-statistic:            {t_welch_raw:.4f}")
print(f"Degrees of freedom:     {v_raw:.2f}")
print(f"Attained p-value:       {p_welch_raw:.6f}  ({p_welch_raw:.3g} to 3 s.f.)")

t_welch_check_raw, p_welch_check_raw = stats.ttest_ind(weekend_pop_raw, weekday_pop_raw, equal_var=False)
print(f"scipy check: t={t_welch_check_raw:.4f}, p={p_welch_check_raw:.10f}  (matches manual calc: {np.isclose(t_welch_raw, t_welch_check_raw)})")

Weekend population (raw): n=2,204, mean=$369.25, sd=$540.61
Weekday population (raw): n=17,755, mean=$554.31, sd=$1875.52

Pooled variance (Sp^2): 3161535.6092
t-statistic:            -4.6085
Degrees of freedom:     19957
Attained p-value:       0.000004  (4.08e-06 to 3 s.f.)
scipy check: t=-4.6085, p=0.0000041  (matches manual calc: True)

Welch's t-test:
t-statistic:            -10.1762
Degrees of freedom:     10731.02
Attained p-value:       0.000000  (3.26e-24 to 3 s.f.)
scipy check: t=-10.1762, p=0.0000000000  (matches manual calc: True)


### Restrict to typical buyers, at the population level

As before, it is worth restricting the population to typical buyers. Trimming the same way as in Step 2, applied to the whole population, gives a picture
of the typical buyer.

In [34]:
q1, q3 = orders["OrderValue"].quantile([0.25, 0.75])
cutoff = q3 + 1.5 * (q3 - q1)
print(f"Q1: ${q1:.2f}   Q3: ${q3:.2f}   Cutoff: ${cutoff:.2f}")

orders_trimmed = orders[orders["OrderValue"] <= cutoff].copy()

weekend_pop = orders_trimmed.loc[orders_trimmed.DayType == "Weekend", "OrderValue"].values
weekday_pop = orders_trimmed.loc[orders_trimmed.DayType == "Weekday", "OrderValue"].values

n1, n2 = len(weekend_pop), len(weekday_pop)
x1, x2 = weekend_pop.mean(), weekday_pop.mean()
s1, s2 = weekend_pop.std(ddof=1), weekday_pop.std(ddof=1)

print(f"Weekend population: n={n1:,}, mean=${x1:.2f}, sd=${s1:.2f}")
print(f"Weekday population: n={n2:,}, mean=${x2:.2f}, sd=${s2:.2f}")

# Pooled test (assumes equal variances)
sp2 = ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)
sp = np.sqrt(sp2)
se = sp * np.sqrt(1/n1 + 1/n2)
t_stat = (x1 - x2) / se
dof = n1 + n2 - 2
p_value = 2 * stats.t.sf(abs(t_stat), dof)

print(f"\nPooled variance (Sp^2): {sp2:.4f}")
print(f"t-statistic:            {t_stat:.4f}")
print(f"Degrees of freedom:     {dof}")
print(f"Attained p-value:       {p_value:.6f}  ({p_value:.3g} to 3 s.f.)")

t_check, p_check = stats.ttest_ind(weekend_pop, weekday_pop, equal_var=True)
print(f"scipy check: t={t_check:.4f}, p={p_check:.4f}  (matches manual calc: {np.isclose(t_stat, t_check)})")

# Welch's test (does not assume equal variances)
se_welch = np.sqrt(s1**2/n1 + s2**2/n2)
t_welch = (x1 - x2) / se_welch
v = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))
p_welch = 2 * stats.t.sf(abs(t_welch), v)

print(f"\nWelch's t-test:")
print(f"t-statistic:            {t_welch:.4f}")
print(f"Degrees of freedom:     {v:.2f}")
print(f"Attained p-value:       {p_welch:.6f}  ({p_welch:.3g} to 3 s.f.)")

t_welch_check, p_welch_check = stats.ttest_ind(weekend_pop, weekday_pop, equal_var=False)
print(f"scipy check: t={t_welch_check:.4f}, p={p_welch_check:.4f}  (matches manual calc: {np.isclose(t_welch, t_welch_check)})")

Q1: $152.49   Q3: $495.50   Cutoff: $1010.01
Weekend population: n=2,117, mean=$298.24, sd=$202.98
Weekday population: n=16,032, mean=$308.31, sd=$219.54

Pooled variance (Sp^2): 47381.6737
t-statistic:            -2.0009
Degrees of freedom:     18147
Attained p-value:       0.045420  (0.0454 to 3 s.f.)
scipy check: t=-2.0009, p=0.0454  (matches manual calc: True)

Welch's t-test:
t-statistic:            -2.1248
Degrees of freedom:     2811.39
Attained p-value:       0.033690  (0.0337 to 3 s.f.)
scipy check: t=-2.1248, p=0.0337  (matches manual calc: True)


## Conclusions

The table below summarizes the findings across all four stages of the analysis using seed 2. Because sampling introduces natural variability, running this script with a different seed or drawing a fresh sample will yield slightly different numerical estimates.

| Stage | n (per group) | mean gap | p (pooled) | p (Welch) |
|---|---|---|---|---|
| Raw sample | 500/500 | $\$89.78$ | 0.01073 | 0.01077 |
| Trimmed sample | 474/444 | $\$13.58$ | 0.3067 | 0.3072 |
| Full population | 2,204/17,755 | $\$185.06$ | 4.08e-06 | 3.26e-24 |
| Full trimmed population | 2,117/16,032 | $\$10.07$ | 0.0454 | 0.0337 |

**Core Question:** Is there sufficient evidence that weekend and weekday shoppers spend differently, and what is the attained significance level?

**Key Takeaways:**

1. **Statistical power vs. absence of effect:** A researcher analyzing only the trimmed sample ($n = 474/444$) would calculate $p \approx 0.31$ and might mistakenly claim that weekend and weekday spending levels are identical. Analyzing the full trimmed population proves that a real difference exists ($p < 0.05$ across both tests). The sample failed to show a difference due to limited sample size, demonstrating why a high $p$-value indicates a lack of statistical power rather than proof of equality.

2. **Directional bias in small samples:** In the trimmed sample, weekend orders averaged higher spending than weekday orders ($\$304.61$ vs. $\$291.03$). In the full trimmed population, this relationship reverses: weekday orders average higher spending ($\$308.31$ vs. $\$298.24$). Relying solely on a small sample risks misidentifying which group spends more.

3. **Sample imbalance and test divergence:** When sample sizes are balanced ($n_1 = n_2 = 500$), pooled and Welch $t$-statistics yield identical values ($t = -2.5561$), differing only in their degrees of freedom. As sample sizes become imbalanced ($n_1 = 474, n_2 = 444$), the test statistics begin to separate ($1.0227$ vs. $1.0218$). At the full population scale ($n_1 \approx 2{,}100$ vs. $n_2 \approx 18{,}000$), the two tests diverge significantly. This divergence signals both extreme sample imbalance and a possible violation of the equal variance assumption required for the pooled $t$-test.

---
#### Notes
* **Core Analysis:** All feature engineering, mathematical derivations, model training, and evaluation were developed by the author.
* **Editing Assistants:** Claude and Gemini were used to help refine text and code formatting.